# Tidy Data와 연구 시각화

## Bike Sharing 자료의 집계와 시각화

시간별 관측을 시간대와 근무일 조건으로 집계하고, 집계 전후의 grain과 그림이 답하는 질문을 확인합니다.


## 이번 실습에서 완성할 것

- 집계 전후 한 행의 의미를 적습니다.
- 평균·중앙값·관측 수를 함께 비교합니다.
- 시간대 그림과 분포 그림의 질문을 구분합니다.

**완료 결과:** 집계표 1개, 그림 2개, 집계함수와 표본 수를 포함한 해석문장


## 이번 교시에서 사용할 데이터

### 강의 시연: OWID CO₂ 국가–연도 표

- **규모:** 175행, 5개 지역, 1990~2024년
- **현재 grain:** 지역 × 연도
- **reshape에 사용할 key:** `code + year`
- **measure:** 총 CO₂ 배출량 `emissions_total`

원자료는 국가가 행에 반복되는 long 구조입니다. `pivot`을 적용하면 연도가 행, 국가가 열인 wide 구조가 됩니다. 구조가 달라져도 같은 `code–year`의 값이 보존되는지 확인합니다.

### 적용 실습: UCI Bike Sharing 시간별 자료

- **규모:** 17,379행, 2011~2012년
- **현재 grain:** 날짜 × 시간
- **집계에 사용할 열:** `hr`, `workingday`, `weekday`
- **measure:** 시간별 총 대여건수 `cnt`

시간대별 평균을 계산하면 grain이 날짜×시간에서 시간 또는 시간×근무일로 바뀝니다. `groupby` 뒤의 한 행이 무엇을 뜻하는지, 평균과 합계 가운데 어떤 집계가 질문에 맞는지 먼저 정합니다. 정규화된 기온·습도 값은 원래 물리 단위와 다르므로 축 label에 단위를 임의로 붙이지 않습니다.


## 이번 실습의 분석 순서

먼저 전체 Bike Sharing 자료에서 시간대별 집계와 그림을 만듭니다. 그다음 `WORKING_DAY_ONLY=True`로 바꾸고 **모든 집계표와 그림이 같은 선택자료 `selected`를 사용하는지** 확인합니다.

**남길 결과:** 평균·중앙값·관측 수가 있는 집계표, 질문이 다른 그림 두 종류, 집계표 관측 수 합과 선택행 수의 일치 검사


## 실습 구간 안내

| 구분 | 수행 범위 |
|---|---|
| **수업 중 필수** | 자료 선택, 집계·기본 그림, grain·완전성 검사 |
| **시간이 남으면** | 평균·중앙값 곡선과 분포 비교 |
| **수업 후 확장(선택)** | 근무일×날씨등급 평균과 표본 수 |

수업 중에는 필수 구간의 결과와 마지막 `PASS`를 먼저 완성합니다. 확장 구간을 실행하지 않아도 필수 셀이 독립적으로 작동하도록 구성되어 있습니다.


## 실행 환경


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행해 주세요."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 1. 시간별 Bike 자료 읽기

날짜를 datetime으로 읽고 원자료의 날짜 × 시간 grain과 key를 확인합니다.


In [ ]:
BIKE_PATH = find_course_data("bike/hour.csv")
bike = pd.read_csv(BIKE_PATH, parse_dates=["dteday"])
analysis_name = "UCI Bike Sharing hourly data"

print("shape:", bike.shape)
print("duplicate date-hour keys:", bike.duplicated(["dteday", "hr"]).sum())
display(bike[["dteday", "hr", "workingday", "weathersit", "cnt"]].head(3))


## 2. groupby로 grain 바꾸기

원자료를 시간 × 근무일 유형별 요약으로 바꿉니다. 출력의 한 행은 더 이상 특정 날짜의 특정 시간이 아닙니다.


In [ ]:
# True로 바꾸면 근무일만 선택합니다.
WORKING_DAY_ONLY = False

selected = bike.loc[bike["workingday"].eq(1)].copy() if WORKING_DAY_ONLY else bike.copy()
hourly_summary = (
    selected.groupby(["hr", "workingday"], as_index=False)
            .agg(mean_rentals=("cnt", "mean"),
                 median_rentals=("cnt", "median"),
                 observations=("cnt", "size"))
)
display(hourly_summary.head(8))
print("summary grain: hour × workingday")
print("\n[집계 전후 비교]")
print("집계 전 행 수:", len(selected), "| 집계 후 행 수:", len(hourly_summary))
sample_hour = 8
print(f"집계 전 {sample_hour}시 관측 예시 — 날짜마다 한 행이 있습니다.")
display(selected.loc[selected["hr"].eq(sample_hour), ["dteday", "hr", "workingday", "cnt"]].head(6))
print(f"집계 후 {sample_hour}시 — 근무일 여부마다 요약값 한 행이 남습니다.")
display(hourly_summary.loc[hourly_summary["hr"].eq(sample_hour)])


## 3. pivot과 두 가지 시각화

요일 × 시간 평균표를 heatmap으로, 근무일별 시간 패턴을 lineplot으로 표현합니다. 같은 데이터라도 질문에 따라 표의 grain과 시각적 encoding이 달라집니다.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
weekday_hour = (
    selected.groupby(["weekday", "hr"], as_index=False)["cnt"].mean()
        .pivot(index="weekday", columns="hr", values="cnt")
)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
sns.lineplot(data=hourly_summary, x="hr", y="mean_rentals", hue="workingday", marker="o", ax=axes[0])
axes[0].set(title="Hourly rental pattern", xlabel="Hour", ylabel="Mean rentals/hour")
sns.heatmap(weekday_hour, cmap="viridis", ax=axes[1], cbar_kws={"label": "Mean rentals/hour"})
axes[1].set(title="Weekday × hour", xlabel="Hour", ylabel="Weekday code")
plt.tight_layout()
plt.show()

result_table = hourly_summary.sort_values("mean_rentals", ascending=False).head(8)
display(result_table)
print("\n[heatmap에 들어간 실제 표]")
print("행은 weekday, 열은 hour, 셀 값은 해당 조합의 평균 대여건수입니다.")
display(weekday_hour.iloc[:4, :8].round(1))


## 시간이 남으면 — 평균·중앙값 곡선과 분포를 함께 보기

시간대별 평균과 중앙값이 다른 위치를 찾고, 근무일·비근무일의 전체 수요분포를 boxplot으로 비교합니다. 그림 아래에 축·단위·집계함수·표본 범위를 포함한 caption 한 문장을 작성하세요.


In [ ]:
mean_median_by_hour = selected.groupby("hr")["cnt"].agg(mean="mean", median="median", observations="size")
mean_median_by_hour["absolute_gap"] = (mean_median_by_hour["mean"] - mean_median_by_hour["median"]).abs()
print("[평균과 중앙값 차이가 큰 시간대]")
display(mean_median_by_hour.nlargest(6, "absolute_gap").round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
mean_median_by_hour[["mean", "median"]].plot(ax=axes[0], marker="o")
axes[0].set(title="Mean and median demand by hour", xlabel="Hour", ylabel="Rentals/hour")
sns.boxplot(data=selected, x="workingday", y="cnt", showfliers=False, ax=axes[1])
axes[1].set(title="Demand distribution by working-day status", xlabel="Working day", ylabel="Rentals/hour")
plt.tight_layout()
plt.show()

distribution_summary = selected.groupby("workingday")["cnt"].agg(observations="size", mean="mean", median="median")
display(distribution_summary.round(2))


optional_distribution_checks = pd.Series({
    "hour counts cover selected rows": int(mean_median_by_hour["observations"].sum()) == len(selected),
    "distribution counts cover selected rows": int(distribution_summary["observations"].sum()) == len(selected),
}, name="passed")
display(optional_distribution_checks.to_frame())
assert optional_distribution_checks.all()
print("추가 집계 검증: PASS")


## 수업 후 확장(선택) — 근무일×날씨등급의 평균과 표본 수

필수 실습을 마친 뒤 수행합니다. 같은 색상 차이라도 관측 수가 적은 그룹에서는 우연한 변동이 클 수 있습니다. 평균 수요 heatmap과 표본 수 heatmap을 나란히 놓고 해석하세요.


In [ ]:
weather_workday_summary = (
    selected.groupby(["workingday", "weathersit"], observed=True)["cnt"]
    .agg(mean_rentals="mean", median_rentals="median", observations="size")
    .reset_index()
)
print("[근무일×날씨등급 요약표: 한 행은 두 범주의 한 조합]")
display(weather_workday_summary.round(2))

mean_grid = weather_workday_summary.pivot(
    index="workingday", columns="weathersit", values="mean_rentals"
)
count_grid = weather_workday_summary.pivot(
    index="workingday", columns="weathersit", values="observations"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sns.heatmap(mean_grid, annot=True, fmt=".1f", cmap="YlGnBu", ax=axes[0])
axes[0].set(title="Mean rentals by working day and weather", xlabel="Weather category", ylabel="Working day")
sns.heatmap(count_grid, annot=True, fmt=".0f", cmap="Greys", ax=axes[1])
axes[1].set(title="Observation count in each group", xlabel="Weather category", ylabel="Working day")
plt.tight_layout()
plt.show()

smallest_group = weather_workday_summary.loc[weather_workday_summary["observations"].idxmin()]
print(
    "가장 작은 그룹:",
    f"workingday={int(smallest_group['workingday'])},",
    f"weathersit={int(smallest_group['weathersit'])},",
    f"n={int(smallest_group['observations'])}",
)
print("평균의 색상 차이는 오른쪽 표본 수와 함께 해석합니다.")


optional_weather_checks = pd.Series({
    "weather counts cover selected rows": int(weather_workday_summary["observations"].sum()) == len(selected),
    "mean and count grids share shape": mean_grid.shape == count_grid.shape,
}, name="passed")
display(optional_weather_checks.to_frame())
assert optional_weather_checks.all()
print("확장 집계 검증: PASS")


## 4. grain과 완전성 검사

집계 결과가 예상한 시간 범주를 포함하는지, 원자료 key와 관계식이 유지되는지 확인합니다.


In [ ]:
checks = pd.Series({
    "selected key unique": selected.duplicated(["dteday", "hr"]).sum() == 0,
    "24 selected hour categories": set(selected["hr"]) == set(range(24)),
    "selected categories match setting": set(hourly_summary["workingday"]) == ({1} if WORKING_DAY_ONLY else {0, 1}),
    "selected count identity": selected["cnt"].eq(selected["casual"] + selected["registered"]).all(),
    "hourly counts cover selected rows": int(hourly_summary["observations"].sum()) == len(selected),
    "summary nonempty": len(result_table) == 8,
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("구조 변환 검증: PASS")


## 결과 기록과 ChatGPT 검토

다음 셀은 `WORKING_DAY_ONLY`, `hourly_summary`, `result_table`, 필수 `checks`를 직접 넣어 검토 프롬프트를 만듭니다. 선택 확장을 실행하지 않아도 작동합니다.

- **관찰:** 평균·중앙값·관측 수와 그림에서 직접 확인한 패턴
- **조건:** 선택자료, grain, 집계함수, 축과 단위
- **한계:** 작은 표본 그룹과 현재 그림만으로 설명할 수 없는 원인


In [ ]:
required_summary_for_prompt = hourly_summary.sort_values(
    "mean_rentals", ascending=False
).head(8)
review_prompt = f"""
Bike Sharing 집계와 시각화를 다음 실제 출력만 사용해 검토해 주세요.

- WORKING_DAY_ONLY: {WORKING_DAY_ONLY}
- 선택자료 행 수: {len(selected)}
- 집계 grain: hour × workingday
- 시간대별 집계 상위 행:
{required_summary_for_prompt.to_string(index=False)}
- 필수 완전성 검사:
{checks.to_string()}

관찰 | 분석조건 | 해석한계의 세 항목으로 답해 주세요. 집계표 observations의
합과 선택자료 행 수가 같은지 확인하고, 평균·중앙값·관측 수의 역할을 구분해
주세요. 실행하지 않은 선택 확장의 결과는 추측하지 마세요.
""".strip()
print("[ChatGPT에 복사할 교시별 검토 프롬프트]\n")
print(review_prompt)


## 완료 확인

마지막 출력이 `구조 변환 검증: PASS`인지 확인해 주세요. 이 검사는 필요한 계산이 실행됐는지 확인할 뿐, 연구 주장의 타당성을 자동 판정하지 않습니다.


## 결과를 정리할 질문

- 원자료의 한 행과 집계표의 한 행은 어떻게 다릅니까?
- 평균과 중앙값의 차이가 큰 시간대는 어디이며, 그 그룹의 관측 수는 얼마입니까?
- 선택한 그림에서 직접 관찰한 패턴과 현재 그림만으로 설명할 수 없는 원인을 구분했습니까?
